# H1+ZEUS NC inclusive data → pandas + **per-$Q^2$** baseline + GCFT **xion** prior (illustrative)

Public data: **Aaron et al., JHEP 01 (2010) 109** — combined H1 and ZEUS inclusive NC $e^+p$ on [HEPData ins836107](https://www.hepdata.net/record/ins836107). Tables list **reduced cross section** $\sigma_r$ (many `%` uncertainty columns) and **$F_2$** where quoted.

**This notebook:** merges $\sigma_r$ with $F_2$, builds **stat** and **quadrature-combined** relative errors from the CSV, fits **global** and **per-$Q^2$** baselines (quadratic in $\log x$ when $n\geq 4$), samples an illustrative **xion** prior, optional **MAP** on $\eta$, loads **HERA 2015** $\sigma_r$ ([ins1377206](https://www.hepdata.net/record/ins1377206)) for a second-era coverage plot, and points to **YAML** bundles for full covariances.

**Still elsewhere:** dedicated $F_L$ tables, diffractive cross sections, CC — add other HepData record IDs with the same parsing style.

**Open in Colab:** [GitHub → Colab](https://colab.research.google.com/) → paste `ChaonickGCFT/Zeus_Notebook` / `notebooks/gcft_hera_f2_xion_prior.ipynb`.

**Xion** = small multiplicative bump in $\log x$ with Gaussian priors — a **scaffold**, not a published GCFT likelihood.

In [ ]:
import csv
import io
import re
import subprocess
import sys
import tarfile
import gzip
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:

    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

HEPDATA_CSV_TGZ = "https://www.hepdata.net/download/submission/ins836107/1/csv"
HERA15_CSV_TGZ = "https://www.hepdata.net/download/submission/ins1377206/1/csv"

try:
    import yaml  # noqa: F401 — optional; parse HEPData YAML tables locally
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml  # noqa: F401

try:
    import scipy.optimize as spo
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    import scipy.optimize as spo

# Bump when you push substantive changes (helps spot stale Colab/GitHub cache).
NOTEBOOK_REV = "2026-04-03 — quad per-Q², σ comb (CSV), HERA15 σ_r, MAP xion, yaml hook; np.vstack"
print(NOTEBOOK_REV)

In [ ]:
def _parse_pct(tok: str) -> float:
    tok = tok.strip().strip('"').rstrip("%")
    return float(tok) / 100.0


def load_hepdata_ins836107_bundle(url: str = HEPDATA_CSV_TGZ) -> pd.DataFrame:
    """Parse each Table*.csv: reduced cross section block + $F_2$ block; merge on $(Q^2,x,y,E_{\rm cm})$."""
    raw = urlopen(url).read()
    sig_rows, f2_rows = [], []
    with tarfile.open(fileobj=gzip.GzipFile(fileobj=io.BytesIO(raw)), mode="r|") as tar:
        for member in tar:
            if not member.isfile() or not member.name.endswith(".csv"):
                continue
            text = tar.extractfile(member).read().decode("utf-8", errors="replace")
            q2 = None
            mode = None
            for line in text.splitlines():
                line = line.strip()
                if not line:
                    continue
                m = re.match(r"#:\s*Q\*\*2\s*\[GeV\^2\]\s*,\s*([0-9.eE+-]+)", line)
                if m:
                    q2 = float(m.group(1))
                    mode = None
                    continue
                if line.startswith("#"):
                    continue
                ul = line.upper()
                if ul.startswith("X,") and "SIG" in ul and "REDUCED" in ul.replace(" ", ""):
                    mode = "sig"
                    continue
                if ul.startswith("X,") and "F2" in ul and "SIG" not in ul:
                    mode = "f2"
                    continue
                if mode == "sig" and q2 is not None:
                    rdr = csv.reader([line])
                    parts = next(rdr)
                    parts = [p.strip() for p in parts]
                    if len(parts) < 6:
                        continue
                    try:
                        x, y, ecm = float(parts[0]), float(parts[1]), float(parts[2])
                        sig = float(parts[3])
                    except ValueError:
                        continue
                    rel_parts = []
                    i = 4
                    while i + 1 < len(parts):
                        try:
                            dp = abs(_parse_pct(parts[i]))
                            dm = abs(_parse_pct(parts[i + 1]))
                            rel_parts.append(0.5 * (dp + dm))
                        except ValueError:
                            break
                        i += 2
                    stat_rel = rel_parts[0] if rel_parts else 0.0
                    comb_rel = float(np.sqrt(np.sum(np.square(rel_parts)))) if rel_parts else stat_rel
                    sig_rows.append(
                        {
                            "source": member.name,
                            "Q2": q2,
                            "x": x,
                            "y": y,
                            "Ecm": ecm,
                            "sigma_r": sig,
                            "sigma_r_stat_rel": stat_rel,
                            "sigma_r_comb_rel": comb_rel,
                        }
                    )
                elif mode == "f2" and q2 is not None:
                    parts = [p.strip() for p in line.split(",")]
                    if len(parts) < 4:
                        continue
                    try:
                        x, y, ecm = float(parts[0]), float(parts[1]), float(parts[2])
                    except ValueError:
                        continue
                    f2tok = parts[3].strip()
                    if f2tok in ("-", "", "nan"):
                        continue
                    try:
                        f2 = float(f2tok)
                    except ValueError:
                        continue
                    norm_rel = 0.005
                    if len(parts) >= 6 and "%" in parts[4]:
                        try:
                            norm_rel = max(norm_rel, abs(_parse_pct(parts[4])))
                        except ValueError:
                            pass
                    f2_rows.append(
                        {
                            "source": member.name,
                            "Q2": q2,
                            "x": x,
                            "y": y,
                            "Ecm": ecm,
                            "F2": f2,
                            "F2_norm_rel": norm_rel,
                        }
                    )
    sig_df = pd.DataFrame(sig_rows)
    f2_df = pd.DataFrame(f2_rows)
    keys = ["Q2", "x", "y", "Ecm"]
    sig_df = sig_df.drop_duplicates(subset=keys, keep="first")
    f2_df = f2_df.drop_duplicates(subset=keys, keep="first")
    merge_cols = keys + ["sigma_r", "sigma_r_stat_rel", "sigma_r_comb_rel"]
    merged = f2_df.merge(sig_df[merge_cols], on=keys, how="left")
    floor = merged["F2_norm_rel"].clip(lower=0.005)
    merged["F2_rel_err_stat"] = merged["sigma_r_stat_rel"].fillna(floor)
    merged["F2_rel_err_comb"] = merged["sigma_r_comb_rel"].fillna(floor)
    merged["F2_rel_err"] = merged["F2_rel_err_comb"]
    return merged


df = load_hepdata_ins836107_bundle()
df = df[np.isfinite(df["F2"]) & (df["F2"] > 0)].copy()
df["logx"] = np.log(df["x"])
df["logQ2"] = np.log(df["Q2"])
df["logF2"] = np.log(df["F2"])
print(len(df), "$F_2$ points after merge")
print("$Q^2$ range:", df["Q2"].min(), "…", df["Q2"].max(), "GeV$^2$")
print("Matched $\sigma_r$ rows:", df["sigma_r"].notna().sum(), "/", len(df))
print(
    "Median rel.~err (stat vs comb, %):",
    round(100 * df["F2_rel_err_stat"].median(), 3),
    ",",
    round(100 * df["F2_rel_err"].median(), 3),
)
df.head()

### Kinematic coverage and $Q^2$ slice picker

Each HEPData **table** is one fixed $Q^2$. The next cell summarises bins and defines **`pick_q2_slices`**: it chooses four $Q^2$ values spaced by **cumulative point count** (so you see bins with real statistics, not just the first/last $Q^2$ in sorted order).

In [ ]:
q2_bins = (
    df.groupby("Q2", as_index=False)
    .agg(
        n_F2=("F2", "count"),
        x_min=("x", "min"),
        x_max=("x", "max"),
        F2_median=("F2", "median"),
    )
    .sort_values("Q2")
)
print("Distinct Q² bins:", len(q2_bins), "| total F₂ points:", int(q2_bins["n_F2"].sum()))
display(q2_bins.head(6))
display(q2_bins.tail(6))


def pick_q2_slices(d: pd.DataFrame, k: int = 4):
    """Pick k table Q² values by cumulative point mass (spread across the dataset)."""
    vc = d.groupby("Q2").size().reset_index(name="n").sort_values("Q2")
    qs = vc["Q2"].to_numpy(float)
    w = vc["n"].to_numpy(float)
    if len(qs) == 0:
        return []
    if len(qs) <= k:
        out = list(qs)
        while len(out) < k:
            out.append(qs[-1])
        return out[:k]
    total = w.sum()
    cum = np.cumsum(w) / total
    targets = (np.arange(k) + 0.5) / k
    chosen = []
    for t in targets:
        j = int(np.searchsorted(cum, t, side="left"))
        j = min(max(j, 0), len(qs) - 1)
        chosen.append(qs[j])
    uniq = []
    for q in chosen:
        if not any(np.isclose(q, u, rtol=1e-9, atol=1e-12) for u in uniq):
            uniq.append(float(q))
    i = 0
    while len(uniq) < k and i < len(qs):
        c = float(qs[i])
        if not any(np.isclose(c, u, rtol=1e-9, atol=1e-12) for u in uniq):
            uniq.append(c)
        i += 1
    return uniq[:k]


print("Representative Q² (4-way by stats):", [f"{q:g}" for q in pick_q2_slices(df, 4)])

## Baselines

1. **Global plane** — $\log F_2 = c_0 + c_1\log x + c_2\log Q^2$ (original toy).
2. **Global quadratic surface** — adds $(\log x)^2$, $(\log Q^2)^2$, $(\log x)(\log Q^2)$ for curvature.
3. **Per-$Q^2$ surface** — if a bin has $\geq 4$ points: $\log F_2 = a + b\log x + c(\log x)^2$; if $2$–$3$ points: a line in $\log x$; else fall back to the global quadratic surface.
4. **Uncertainties** — from each $\sigma_r$ CSV row, all listed `+` / `-` `%` bands are combined **in quadrature** (treated as independent components — still **not** the full HEPData correlation matrix).

## Xion prior (same as before)

$$
F_2^{\rm pred} = F_2^{\rm base}\,\exp\bigl(\eta\,\phi(x)\bigr),\quad
\phi(x)=\exp\!\left(-\frac{(\log(x/x_0))^2}{2w^2}\right),
$$
with $\eta\sim\mathcal{N}(0,\sigma_\eta^2)$, $\log_{10}x_0$ Gaussian, fixed $w$.

In [ ]:
def fit_global_plane(d: pd.DataFrame):
    X = np.column_stack([np.ones(len(d)), d["logx"], d["logQ2"]])
    coef, _, _, _ = np.linalg.lstsq(X, d["logF2"], rcond=None)
    return coef


def fit_global_quad(d: pd.DataFrame):
    lx, lq = d["logx"].values, d["logQ2"].values
    X = np.column_stack(
        [np.ones(len(d)), lx, lq, lx ** 2, lq ** 2, lx * lq]
    )
    coef, _, _, _ = np.linalg.lstsq(X, d["logF2"], rcond=None)
    return coef


def log_f2_global_plane(x, Q2, coef):
    c0, c1, c2 = coef
    return c0 + c1 * np.log(x) + c2 * np.log(Q2)


def log_f2_global_quad(x, Q2, coef):
    c0, c1, c2, c3, c4, c5 = coef
    lx = np.log(x)
    lq = np.log(Q2)
    return c0 + c1 * lx + c2 * lq + c3 * lx ** 2 + c4 * lq ** 2 + c5 * lx * lq


def fit_per_q2_surfaces(d: pd.DataFrame):
    """dict Q2 -> ('quad', c3) | ('line', c2) | ('none', None) for log F2 models in x."""
    out = {}
    for q2, g in d.groupby("Q2"):
        n = len(g)
        if n < 2:
            out[q2] = ("none", None)
        elif n < 4:
            X = np.column_stack([np.ones(n), g["logx"].values])
            c, _, _, _ = np.linalg.lstsq(X, g["logF2"].values, rcond=None)
            out[q2] = ("line", c)
        else:
            lx = g["logx"].values
            X = np.column_stack([np.ones(n), lx, lx ** 2])
            c, _, _, _ = np.linalg.lstsq(X, g["logF2"].values, rcond=None)
            out[q2] = ("quad", c)
    return out


def freeze_per_q2_above(per_q2: dict, d: pd.DataFrame, q_cut: float = 500.0) -> dict:
    """For Q² > q_cut, reuse the fit from the largest table Q² ≤ q_cut (stabilises high-Q² extrapolation)."""
    uq = np.sort(d["Q2"].unique())
    le = uq[uq <= q_cut]
    q_ref = float(le.max()) if len(le) else float(uq.min())
    ref = None
    for k, v in per_q2.items():
        if np.isclose(float(k), q_ref, rtol=0.0, atol=1e-6):
            ref = v
            break
    if ref is None or ref[1] is None:
        return per_q2
    rk, rc = ref[0], np.array(ref[1], dtype=float, copy=True)
    out = {}
    for k, v in per_q2.items():
        fk = float(k)
        if fk > q_cut:
            out[k] = (rk, rc.copy())
        else:
            if v[1] is None:
                out[k] = v
            else:
                out[k] = (v[0], np.array(v[1], dtype=float, copy=True))
    return out


def log_f2_per_q2(x, Q2, per_q2, coef_quad):
    x = np.asarray(x, dtype=float)
    Q2 = np.asarray(Q2, dtype=float)
    out = np.empty_like(x, dtype=float)
    xr, qr, orv = x.ravel(), Q2.ravel(), out.ravel()
    for i in range(xr.size):
        xi, qi = xr[i], qr[i]
        ent = per_q2.get(qi)
        if ent is None:
            ent = per_q2.get(float(qi))
        kind, c = ent if ent is not None else ("none", None)
        if kind == "quad" and c is not None:
            lx = np.log(xi)
            orv[i] = c[0] + c[1] * lx + c[2] * lx ** 2
        elif kind == "line" and c is not None:
            orv[i] = c[0] + c[1] * np.log(xi)
        else:
            orv[i] = log_f2_global_quad(xi, qi, coef_quad)
    return out


coef_plane = fit_global_plane(df)
coef_quad = fit_global_quad(df)
per_q2_raw = fit_per_q2_surfaces(df)
FREEZE_Q2_CUT = 500.0
per_q2 = freeze_per_q2_above(per_q2_raw, df, FREEZE_Q2_CUT)
print(
    f"Per-Q² fits: Q² > {FREEZE_Q2_CUT:g} GeV² reuse coefficients from max table Q² ≤ cut (freeze patch)."
)


def phi_xion(x, x0, w):
    return np.exp(-0.5 * (np.log(x / x0) / w) ** 2)


def f2_from_log(logf2, eta, phi):
    return np.exp(logf2 + eta * phi)


rng = np.random.default_rng(42)
sigma_eta = 0.04
mu_log10_x0, tau_log10_x0 = -3.0, 0.35
w_xion = 1.15
n_draw = 400

etas = rng.normal(0.0, sigma_eta, size=n_draw)
log10_x0s = rng.normal(mu_log10_x0, tau_log10_x0, size=n_draw)
x0s = 10.0 ** log10_x0s

xv = df["x"].values
Q2v = df["Q2"].values
f2v = df["F2"].values
sig_rel = df["F2_rel_err"].values
sig_rel_stat = df["F2_rel_err_stat"].values

log_plane = log_f2_global_plane(xv, Q2v, coef_plane)
log_perq = log_f2_per_q2(xv, Q2v, per_q2, coef_quad)


def residual_samples(log_base):
    cols = []
    for eta, x0 in zip(etas, x0s):
        phi = phi_xion(xv, x0, w_xion)
        fp = f2_from_log(log_base, eta, phi)
        cols.append((f2v - fp) / f2v)
    return np.column_stack(cols)


rs_plane = residual_samples(log_plane)
rs_perq = residual_samples(log_perq)

med_p = np.median(rs_plane, axis=1)
med_q = np.median(rs_perq, axis=1)
pull_p = med_p / sig_rel
pull_q = med_q / sig_rel
pull_q_statonly = med_q / np.maximum(sig_rel_stat, 1e-15)

# Baseline quality **without** xion (exp(log μ)); proxy Gaussian-ish χ² / ndf using stat errors only
pred_plane = np.exp(np.clip(log_plane, -80.0, 80.0))
pred_perq = np.exp(np.clip(log_perq, -80.0, 80.0))
sig_f2 = np.maximum(sig_rel * f2v, 1e-15)
rmse_rel_plane = float(np.sqrt(np.mean(((f2v - pred_plane) / f2v) ** 2)))
rmse_rel_perq = float(np.sqrt(np.mean(((f2v - pred_perq) / f2v) ** 2)))
rmse_log_plane = float(np.sqrt(np.mean((np.log(f2v) - log_plane) ** 2)))
rmse_log_perq = float(np.sqrt(np.mean((np.log(f2v) - log_perq) ** 2)))
sig_f2_stat = np.maximum(sig_rel_stat * f2v, 1e-15)
ms_pull2_plane = float(np.mean(((f2v - pred_plane) / sig_f2) ** 2))
ms_pull2_perq = float(np.mean(((f2v - pred_perq) / sig_f2) ** 2))
ms_pull2_plane_st = float(np.mean(((f2v - pred_plane) / sig_f2_stat) ** 2))
ms_pull2_perq_st = float(np.mean(((f2v - pred_perq) / sig_f2_stat) ** 2))
print("— Baselines (no xion) —")
print(f"  RMSE_rel  global plane vs per-Q² : {rmse_rel_plane:.4f}  vs  {rmse_rel_perq:.4f}")
print(f"  RMSE_logF2 global plane vs per-Q² : {rmse_log_plane:.4f}  vs  {rmse_log_perq:.4f}")
print(f"  mean(pull²) per-Q², σ = comb CSV : {ms_pull2_perq:.2f}  (global plane {ms_pull2_plane:.2f})")
print(f"  mean(pull²) per-Q², σ = stat only: {ms_pull2_perq_st:.2f}  (global plane {ms_pull2_plane_st:.2f})")

# --- MAP on xion amplitude η at fixed x₀ (Gaussian prior + diagonal χ²) ---
x0_map = 10.0 ** mu_log10_x0
phi_map = phi_xion(xv, x0_map, w_xion)


def neg_log_post_eta(eta, log_mu, f2, sig_rel_abs, phi, s_eta):
    pred = np.exp(np.clip(log_mu + eta * phi, -80.0, 80.0))
    den = np.maximum(sig_rel_abs * f2, 1e-15)
    chi2 = np.sum(((f2 - pred) / den) ** 2)
    return 0.5 * chi2 + 0.5 * (eta / s_eta) ** 2


res_map = spo.minimize_scalar(
    lambda e: neg_log_post_eta(e, log_perq, f2v, sig_rel, phi_map, sigma_eta),
    bounds=(-0.25, 0.25),
    method="bounded",
)
print(
    f"— MAP η (x₀=fixed prior mean, per-Q² quad baseline, σ=comb): {res_map.x:.5f} —"
)
print(f"    min neg-log-posterior value: {res_map.fun:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True)
for ax, med, title in zip(
    axes,
    [med_p, med_q],
    [
        "global log-plane baseline",
        "per-$Q^2$ quad/line in $\log x$ (+ global quad fallback)",
    ],
):
    sc = ax.scatter(Q2v, med, c=np.log10(xv), s=10, alpha=0.72, cmap="viridis")
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xscale("log")
    ax.set_xlabel(r"$Q^2\;(GeV^2)$")
    ax.set_ylabel(r"median rel.~res. $(F_2^{data}-F_2^{pred})/F_2^{data}$")
    ax.set_title(title + " + xion prior draws")
    lo, hi = np.nanpercentile(med, [2, 98])
    pad = max(0.05, 0.15 * (hi - lo))
    ax.set_ylim(lo - pad, hi + pad)

fig.suptitle("H1+ZEUS combined NC $F_2$: baseline comparison (xion median over prior)", y=1.02)
fig.colorbar(sc, ax=axes, label=r"$\log_{10}(x)$", fraction=0.02, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
pq = pull_q[np.isfinite(pull_q)]
ps = pull_q_statonly[np.isfinite(pull_q_statonly)]
ax.hist(pq, bins=36, color="steelblue", alpha=0.65, edgecolor="white", label=r"σ = comb (stat+sys in quad.)")
ax.hist(ps, bins=36, color="darkorange", alpha=0.45, edgecolor="white", label=r"σ = stat only")
ax.axvline(0.0, color="k", ls="--", lw=0.9)
ax.set_xlabel(r"pull (median rel.~residual / $\hat\sigma_{F_2}$)")
ax.set_ylabel("counts")
ax.set_title("Per-$Q^2$ quad baseline + xion median: pull distributions")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
q_demo = pick_q2_slices(df, 4)[1]
sub_sr = df[np.isclose(df["Q2"], q_demo)]
axes[0].errorbar(
    sub_sr["x"],
    sub_sr["sigma_r"],
    yerr=sub_sr["sigma_r_comb_rel"] * sub_sr["sigma_r"],
    fmt="o",
    ms=3,
    capsize=2,
    color="0.25",
    alpha=0.85,
)
axes[0].set_xscale("log")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$\sigma_r$ (reduced)")
axes[0].set_title(rf"Reduced cross section vs $x$ at $Q^2 = {q_demo:g}$ GeV$^2$")

sc = axes[1].scatter(
    df["Q2"],
    df["x"],
    c=np.log10(df["F2"]),
    s=16,
    alpha=0.55,
    cmap="cividis",
    edgecolors="none",
)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"$Q^2$ (GeV$^2$)")
axes[1].set_ylabel(r"$x$")
axes[1].set_title(r"Coverage in $(Q^2, x)$ ($\log_{10} F_2$ colour)")
plt.colorbar(sc, ax=axes[1], label=r"$\log_{10} F_2$")
plt.tight_layout()
plt.show()

### Secondary set: **HERA 2015** combined NC $\sigma_r$ ([ins1377206](https://www.hepdata.net/record/ins1377206))

Independent of the 2010 $F_2$ tables: each CSV row is $(Q^2, x, \sigma_r)$ with the same **quadrature** combination of listed `%` uncertainties. Use it for coverage / cross-era checks — **not** naively merged with ins836107 without a consistent theory interface.

In [ ]:
def load_hepdata_sigma_long_rows(url: str) -> pd.DataFrame:
    """CSV rows whose first three fields parse as Q², x, σ (2015-style tables)."""
    raw = urlopen(url).read()
    rows = []
    with tarfile.open(fileobj=gzip.GzipFile(fileobj=io.BytesIO(raw)), mode="r|") as tar:
        for member in tar:
            if not member.isfile() or not member.name.endswith(".csv"):
                continue
            for line in tar.extractfile(member).read().decode("utf-8", errors="replace").splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                parts = next(csv.reader([line]))
                parts = [p.strip() for p in parts]
                if len(parts) < 6:
                    continue
                try:
                    q2v, xv, sig = float(parts[0]), float(parts[1]), float(parts[2])
                except ValueError:
                    continue
                rels = []
                i = 3
                while i + 1 < len(parts):
                    try:
                        rels.append(
                            0.5
                            * (abs(_parse_pct(parts[i])) + abs(_parse_pct(parts[i + 1])))
                        )
                    except ValueError:
                        break
                    i += 2
                cr = float(np.sqrt(np.sum(np.square(rels)))) if rels else 0.01
                rows.append(
                    {"Q2": q2v, "x": xv, "sigma_r": sig, "sigma_comb_rel": cr, "table": member.name}
                )
    return pd.DataFrame(rows)


df15 = load_hepdata_sigma_long_rows(HERA15_CSV_TGZ)
print("HERA 2015 σ_r rows:", len(df15), "| Q² range:", df15["Q2"].min(), "…", df15["Q2"].max())

fig, ax = plt.subplots(figsize=(7, 4.2))
sc = ax.scatter(
    df15["Q2"],
    df15["x"],
    c=np.log10(df15["sigma_r"].clip(lower=1e-6)),
    s=10,
    alpha=0.5,
    cmap="plasma",
    edgecolors="none",
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$Q^2$ (GeV$^2$)")
ax.set_ylabel(r"$x$")
ax.set_title(r"HERA 2015 combined NC: $(Q^2,x)$ coverage ($\log_{10}\sigma_r$ colour)")
plt.colorbar(sc, ax=ax, label=r"$\log_{10}\sigma_r$")
plt.tight_layout()
plt.show()

### HEPData **YAML** (correlations / full systematics)

Diagonal **comb** errors above are not a covariance. For a publication-grade fit, download the YAML bundle and build $\mathbf{V}$ (or use tools that consume HEPData JSON/YAML directly):

- [ins836107 YAML](https://www.hepdata.net/download/submission/ins836107/1/yaml) (this $F_2$ / $\sigma_r$ combination)
- [ins1377206 YAML](https://www.hepdata.net/download/submission/ins1377206/1/yaml) (2015 combination)

`pyyaml` is imported in the first cell so you can `yaml.safe_load` individual table files inside those archives in a follow-on notebook.

In [ ]:
YAML_2010 = "https://www.hepdata.net/download/submission/ins836107/1/yaml"
YAML_2015 = "https://www.hepdata.net/download/submission/ins1377206/1/yaml"
print("YAML bundles (tar.gz inside browser / wget):\n ", YAML_2010, "\n ", YAML_2015)

In [ ]:
chosen = pick_q2_slices(df, 4)

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=False, sharey=False)
axes = axes.ravel()

for ax, q2 in zip(axes, chosen):
    sub = df[np.isclose(df["Q2"], q2)]
    # Draw smooth curves only over a **central** x window (default 6–94% of points).
    # If one point sits at tiny x and the rest at large x, a log-x line from xmin→xmax
    # still blows up in between; trimming the plotted x range avoids a fake spike.
    lx = np.log10(sub["x"].to_numpy())
    lo, hi = np.percentile(lx, [6, 94])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        lo, hi = float(lx.min()), float(lx.max())
    xx_fine = np.logspace(lo, hi, 120)
    ax.errorbar(
        sub["x"],
        sub["F2"],
        yerr=sub["F2_rel_err"] * sub["F2"],
        fmt="o",
        ms=4,
        capsize=2,
        color="0.2",
        alpha=0.85,
        label="data",
    )
    lb = log_f2_per_q2(xx_fine, np.full_like(xx_fine, q2), per_q2, coef_quad)
    f2_line = np.exp(np.clip(lb, -80.0, 80.0))
    ax.plot(xx_fine, f2_line, color="C1", lw=1.8, clip_on=True, label="per-$Q^2$ quad/line")
    curves = []
    for eta, x0 in list(zip(etas, x0s))[:80]:
        phi = phi_xion(xx_fine, x0, w_xion)
        curves.append(np.exp(np.clip(lb + eta * phi, -80.0, 80.0)))
    arr = np.vstack(curves)
    ax.fill_between(
        xx_fine,
        np.percentile(arr, 16, axis=0),
        np.percentile(arr, 84, axis=0),
        color="C0",
        alpha=0.25,
        clip_on=True,
        label="xion 68% (subset)",
    )
    # Y-limits from **data only** (ignore model). If mod_hi was NaN, Python's max()
    # became NaN and set_ylim was ignored → Matplotlib autoscale to ~1e19.
    f2_obs = sub["F2"].to_numpy(dtype=float)
    f2_err = (sub["F2_rel_err"] * sub["F2"]).to_numpy(dtype=float)
    hi_pts = np.nanmax(f2_obs + f2_err)
    hi_obs = np.nanmax(f2_obs)
    y_hi = float(np.nanmax([hi_pts * 1.12, hi_obs * 1.25, 1e-12]))
    if len(f2_obs) >= 3:
        y_hi = max(y_hi, float(np.nanpercentile(f2_obs, 95) * 1.6))
    if not np.isfinite(y_hi) or y_hi <= 0:
        y_hi = 1.0
    ax.set_ylim(0.0, y_hi)
    x_lo, x_hi = float(lx.min()), float(lx.max())
    xpad = 0.05 * max(x_hi - x_lo, 1e-6)
    ax.set_xlim(10 ** (x_lo - xpad), 10 ** (x_hi + xpad))
    ax.set_xscale("log")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$F_2$")
    ax.set_title(rf"$Q^2 = {q2:g}$ GeV$^2$ ({len(sub)} pts)")
    ax.legend(fontsize=8, loc="best")

fig.suptitle("$F_2$ vs $x$ at four $Q^2$ slices (illustrative xion band)", y=1.01)
plt.tight_layout()
plt.show()

### Quick checklist

- **`q2_bins` + `pick_q2_slices`** — point-weighted $Q^2$ choices for the four $F_2(x)$ panels.
- **Per-$Q^2$ model** — quadratic in $\log x$ if $\geq 4$ points in the bin, else line, else global quad surface.
- **`F2_rel_err`** — **combined** stat+systematics in **quadrature** from each $\sigma_r$ row’s `%` pairs (independent-component approximation). **`F2_rel_err_stat`** keeps stat only. Pull histogram overlays both.
- **RMSE / mean(pull²)** — printed for global plane vs per-$Q^2$ baseline **without** xion; **MAP $\eta$** is a crude diagonal-Gaussian posterior mode at fixed $x_0$.
- **$F_2(x)$ panels** — central $\log x$ window for curves; **$y$-limits from data**; $\log F_2$ **clipped** for plotting.
- **`df15`** — HERA **2015** $\sigma_r$ only (ins1377206); not merged with 2010 $F_2$.
- **YAML links** — for real fits, build covariances from the official bundles (not done automatically here).
- **Next:** $F_L$ / diffractive / CC HepData records; YAML→covariance pipeline; replace toy xion with your GCFT likelihood.